# ⚡ AI B-Roll Autopilot — Free GPU Backend

Run Whisper transcription + B-roll matching on a **free Colab T4/A100 GPU**, then connect your local (or hosted) web studio to it.

---

### Before you start
1. `Runtime` → `Change runtime type` → select **T4 GPU** → `Save`
2. Add your API keys to **Colab Secrets** (🔑 lock icon in left sidebar):
   - `GEMINI_API_KEY` — [get one here](https://aistudio.google.com/apikey)
   - `PEXELS_API_KEY` — [get one here](https://www.pexels.com/api/) *(optional)*
   - `GIPHY_API_KEY` — [get one here](https://developers.giphy.com/) *(optional)*
3. Run all cells top-to-bottom. Copy the **Public URL** from Cell 3 output.

> 💡 **Tip:** The tunnel URL changes every Colab session.

In [ ]:
# @title 🚀 1. Setup — Clone Repo & Install Dependencies
import os, sys

REPO_URL  = "https://github.com/Sh1sh1R017/stockpile.git"
BRANCH    = "sorry-i-fucked-up"
CLONE_DIR = "/content/stockpile"

# ── GPU check ────────────────────────────────────────────────────────────────
print("🔍 Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo '⚠️  No GPU — switch to T4 GPU runtime!'

# ── Clone or pull ─────────────────────────────────────────────────────────────
print("\n📦 Setting up repo...")
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print("   Already cloned — pulling latest changes...")
    !git -C {CLONE_DIR} fetch -q origin {BRANCH}
    !git -C {CLONE_DIR} reset -q --hard origin/{BRANCH}
else:
    !rm -rf {CLONE_DIR}
    !git clone -q -b {BRANCH} {REPO_URL} {CLONE_DIR}

%cd /content/stockpile
print("✅ Repo ready.")

In [ ]:
%%capture install_log
# @title    Install system packages & Python dependencies (output captured)
import subprocess, sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "pycloudflared", "uvicorn[standard]", "python-multipart", "nest-asyncio",
], check=True)

In [ ]:
# @title    Verify installation
import shutil
print("✅ Packages installed!")
for tool in ["ffmpeg", "python3"]:
    p = shutil.which(tool)
    print(f"  {'✅' if p else '❌'} {tool}: {p or 'NOT FOUND'}")

In [ ]:
# @title 🔑 2. Configure API Keys
# ─────────────────────────────────────────────────────────────────────────────
# RECOMMENDED: Add keys to Colab Secrets (🔑 left sidebar) so they load
# automatically. The form fields below are a manual fallback.
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

# @markdown ### Manual fallbacks (only if NOT using Colab Secrets)
GEMINI_API_KEY  = ""  # @param {type:"string"}
PEXELS_API_KEY  = ""  # @param {type:"string"}
GIPHY_API_KEY   = ""  # @param {type:"string"}

# @markdown ### Whisper settings
WHISPER_MODEL        = "medium"   # @param ["small", "medium", "large", "large-v3"]
WHISPER_COMPUTE_TYPE = "float32"  # @param ["float32", "float16"]
# float32 = safe on T4 (avoids precision errors); float16 = faster on A100

def _get_secret(name, fallback=""):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val: return val.strip()
    except Exception:
        pass
    return fallback.strip()

gemini_key = _get_secret("GEMINI_API_KEY",  GEMINI_API_KEY)
pexels_key = _get_secret("PEXELS_API_KEY",  PEXELS_API_KEY)
giphy_key  = _get_secret("GIPHY_API_KEY",   GIPHY_API_KEY)

env_lines = [
    "LOCAL_INPUT_FOLDER=input",
    "LOCAL_OUTPUT_FOLDER=output",
    f"WHISPER_MODEL={WHISPER_MODEL}",
    f"WHISPER_COMPUTE_TYPE={WHISPER_COMPUTE_TYPE}",
]
if gemini_key: env_lines.append(f"GEMINI_API_KEY={gemini_key}")
if pexels_key: env_lines.append(f"PEXELS_API_KEY={pexels_key}")
if giphy_key:  env_lines.append(f"GIPHY_API_KEY={giphy_key}")

Path("/content/stockpile/.env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")

print("✅ .env written!")
print(f"   GEMINI  : {'✅ set' if gemini_key else '❌ missing — AI director disabled'}")
print(f"   PEXELS  : {'✅ set' if pexels_key else '⚠️  not set'}")
print(f"   GIPHY   : {'✅ set' if giphy_key  else '⚠️  not set'}")
print(f"   Whisper : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")

In [ ]:
# @title 🌐 3. Launch GPU Backend + Public Tunnel
import os, sys, threading, time, subprocess, urllib.request, nest_asyncio
from pycloudflared import try_cloudflare

nest_asyncio.apply()
os.chdir("/content/stockpile")
PORT = 8000

# ── Start FastAPI server ──────────────────────────────────────────────────────
_log = open("/tmp/backend.log", "w")
_proc = subprocess.Popen(
    [sys.executable, "-m", "ai_broll_autopilot.cli", "serve",
     "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=_log, stderr=subprocess.STDOUT,
)

# ── Wait for /health ─────────────────────────────────────────────────────────
print("⏳ Waiting for backend", end="", flush=True)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print(" ✅", flush=True); break
    except Exception:
        print(".", end="", flush=True); time.sleep(2)
else:
    print("\n❌ Backend didn't start — check logs:")
    !tail -30 /tmp/backend.log
    raise RuntimeError("Backend failed to start")

# ── Open tunnel ───────────────────────────────────────────────────────────────
print("🌐 Opening Cloudflare tunnel...", flush=True)
tunnel = try_cloudflare(port=PORT)
# tunnel.tunnel is the public HTTPS URL string
PUBLIC_URL = tunnel.tunnel

print("\n" + "═" * 65)
print("🚀  AI B-ROLL AUTOPILOT — GPU BACKEND READY")
print("═" * 65)
print(f"  🔗 Public URL  : {PUBLIC_URL}")
print(f"  🤖 Whisper     : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")
print("═" * 65)
print("")
print("📋 TO CONNECT YOUR WEB STUDIO:")
print("  Option A — PowerShell before npm run dev:")
print(f'    $env:BACKEND_API_URL="{PUBLIC_URL}"; npm run dev')
print("")
print("  Option B — write to web/.env.local:")
print(f"    BACKEND_API_URL={PUBLIC_URL}")
print("")
print("⚠️  URL changes every Colab session.")
print("═" * 65)
print("\n🟢 Running. Stop cell (■) to shut down.\n")

heartbeat = 0
while True:
    time.sleep(60)
    heartbeat += 1
    print(f"  💓 [{heartbeat}m] {PUBLIC_URL}", flush=True)

In [ ]:
# @title 🛠️ [Optional] View backend logs if something went wrong
!tail -50 /tmp/backend.log

In [ ]:
# @title 🛠️ [Optional] Health check
import urllib.request, json
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/health") as r:
        print("✅ Backend healthy:", json.loads(r.read()))
except Exception as e:
    print("❌ Backend not reachable:", e)